# DocIntel: Document Layout Analysis

**arXiv ref:** *PARL: Position-Aware Relation Learning Network for Document Layout Analysis* — arxiv:2601.07620, Jan 2026

**PILOT:** *Promptable Interleaved Layout-aware OCR Transformer* — arxiv:2504.03621

This notebook demonstrates pixel-level document layout segmentation using **DeepLabV3** with 7 layout classes.


In [ ]:
from __future__ import annotations
import sys, os; sys.path.insert(0, os.path.abspath('.'))
import torch
import numpy as np
import matplotlib.pyplot as plt
from src.data import make_synthetic, create_dataloaders, LAYOUT_CLASSES, NUM_CLASSES
from src.model import build_layout_model
from src.core import compute_layout_metrics
from src.visualizations import plot_sample, plot_per_class_metrics


In [ ]:
# Generate synthetic document data
data = make_synthetic(n=100, seed=42)
print(f"Documents: {data['n_samples']}")
print(f"Layout classes: {data['class_names']}")
print(f"Image shape: {data['images'].shape}")


# Show a document with its layout mask
fig = plot_sample(data['images'][0], data['masks'][0])
plt.show()


In [ ]:
# Build layout model (DeepLabV3 ResNet-50)
model = build_layout_model(num_classes=NUM_CLASSES)
print(f"Model: {sum(p.numel() for p in model.parameters()):,} params")
print("Architecture: DeepLabV3-ResNet50 with atrous spatial pyramid pooling")


In [ ]:
# Forward pass
with torch.no_grad():
    out = model(data['images'][:4])['out']
    preds = out.argmax(dim=1)
print(f"Output: {out.shape}")
print(f"Predictions: {preds.shape}")


In [ ]:
# Per-class IoU
metrics = compute_layout_metrics(preds.numpy(), data['masks'][:4].numpy(), NUM_CLASSES)
print(f"mIoU: {metrics['miou']:.4f}")
print(f"Pixel Acc: {metrics['pixel_acc']:.4f}")
for i, name in enumerate(LAYOUT_CLASSES):
    print(f"  {name:15s} IoU={metrics['per_class_iou'][i]:.4f}")


In [ ]:
print('DocIntel notebook complete. Document layout analysis enables automated document understanding pipelines.')
